# Stage 0 · one Lego training camera
Attach the supplied **stage0-input.zip** as a private Kaggle input and select the T4 accelerator. Run these three code cells. This notebook finishes after 256 optimizer updates. The RGB strip shows the target, prediction, and accumulated opacity from left to right.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, tempfile, zipfile
import torch

inputs = Path("/kaggle/input")
work = Path("/kaggle/working/recovery-output")
work.mkdir(parents=True, exist_ok=True)
markers = list(inputs.rglob("RECOVERY_STAGE0.txt"))
if len(markers) == 1:
    repo = markers[0].parent
elif not markers:
    kits = list(inputs.rglob("stage0-input.zip"))
    assert len(kits) == 1, "Attach one copy of stage0-input.zip as a notebook input."
    repo = Path(tempfile.mkdtemp(prefix="nerf-stage0-"))
    with zipfile.ZipFile(kits[0]) as archive:
        assert "RECOVERY_STAGE0.txt" in archive.namelist()
        for member in archive.namelist():
            assert (repo / member).resolve().is_relative_to(repo)
        archive.extractall(repo)
else:
    raise ValueError("More than one recovery kit attached. Select one version.")
os.chdir(repo)
manifest = work / "lego.json"
run = work / "stage0-lego"
hardware = {"torch": str(torch.__version__), "cuda": torch.version.cuda,
            "devices": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}
print(hardware)
(work / "hardware.json").write_text(json.dumps(hardware, indent=2))
assert torch.cuda.is_available(), "Select GPU T4 in Notebook settings."

In [ ]:
subprocess.run([sys.executable, "-m", "recovery.data", "--scene", "lego",
                "--source", str(repo / "data/blender/lego"), "--output", str(manifest)], cwd=repo, check=True)
frozen = json.loads(manifest.read_text())
assert frozen["data_hash"] == "bdc22418467720a3dbf5555318532caf17c250a05433f8553ae58372f11e89e2"
assert [len(frozen["splits"][s]) for s in ("dense", "sparse", "test")] == [100, 8, 25]

In [ ]:
from IPython.display import display, FileLink
from PIL import Image

cmd = [sys.executable, "-u", "-m", "recovery.training", "--preset", "diagnostic",
       "--manifest", str(manifest), "--run", str(run), "--device", "cuda:0", "--save-every", "64"]
if (run / "checkpoint.pt").is_file():
    cmd.append("--resume")
with (work / "stage0.log").open("a") as log:
    proc = subprocess.Popen(cmd, cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    return_code = proc.wait()

review = Path("/kaggle/working/stage0-review.zip")
with zipfile.ZipFile(review, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for source in [manifest, work / "hardware.json", work / "stage0.log", *sorted(run.glob("*"))]:
        if source.is_file() and not source.name.startswith(".") and not source.name.endswith(".tmp"):
            archive.write(source, source.relative_to(work))
preview = run / "train-000256.png"
if preview.exists():
    display(Image.open(preview).resize((576, 192), Image.Resampling.NEAREST))
gate = run / "gate.json"
if gate.exists():
    print(gate.read_text())
print(f"Download {review} from the Output pane, including when the gate fails.")
os.chdir("/kaggle/working")
display(FileLink(review.name))
if return_code:
    raise RuntimeError(f"Stage 0 stopped with status {return_code}. Send stage0-review.zip for review.")